In [ ]:
import matplotlib.pyplot as plt
from strauss.sonification import Sonification
from strauss.sources import Objects, Events
from strauss import channels
from strauss.score import Score
import numpy as np
from strauss.generator import Synthesizer
import IPython.display as ipd
import glob
import os
import copy
from pathlib import Path
%matplotlib inline

### <u> Mapping Defaults</u> 

Changed the default `map_lims` range from `[0,1]` to `['0%','100%']`: this means by default sound parameters adapt to the offgset and range of the data.

The only exceptions to this are 3D spatial angles (e.g. `'polar'` and `'azimuth'`), which use an absolute unit system due to their cyclic nature. As 3D spatial mapping is relatively advanced, we also institute a new `'pan'` parameter, ranging from fully left to fully right, which can uses the same `['0%','100%']` defaults as other properties.

To demonstrate, we set up a generator:

In [ ]:
synth = Synthesizer()
synth.load_preset('pitch_mapper')
# manually set note properties to get a suitable sound
synth.modify_preset({'note_length':0.03, # hold each note for 0.03 seconds or 30 ms - what if this was 1s?
                         'volume_envelope': {'use':'on', 'A':0.01,'R':0.07}}) # ✏️ Time to fade out once note is released, using 100 ms

and some fake data, notably with offset and scaling relative to a `[0,1]` range

In [ ]:
yoffset = -700
xscale = 100

x = np.linspace(0,xscale,200)
y =np.log10( np.sin(60*x/scale) +1.01) + yoffset

and then use an `Event` sonification, with a periodic `y` value mapped to `pitch` and additionally mapping `x` to `pan`. This shows a sonification that is unaffected by offsets or scalings of the data:

In [ ]:
system = "stereo"
notes = [["C3","D3","E3","G3","B3","C4","D4","E4","G4","B4","C5","D5","E5","G5","B5"]]
score =  Score(notes, 5, pitch_binning='uniform')

maps = {'pitch':y,
        'time': x,
         'pan': x}

# set up source
sources = Events(maps.keys())
sources.fromdict(maps)
sources.apply_mapping_functions()

soni = Sonification(score, sources, copy.copy(synth), system)
soni.render()
dobj = soni.notebook_display(show_waveform=0)

While manually setting the old `strauss` defaults for `map_lims` (`[0,1]`) ***is*** affected by the offset and scale choices:

In [ ]:
lims = {'time': [0,1], 'pitch': [0,1], 'pan': [0,1]}
soni = Sonification(score, sources, copy.copy(synth), system)
sources.apply_mapping_functions(map_lims = lims)
soni.render()
dobj = soni.notebook_display(show_waveform=0)

for spatial angles, we instead assume an absolute unit system. Replacing pan with spatial angles, we remake the sonification. By default this is in 'cycles' (0-1 represents a whole circle), so we rescale azimuth values to this range:

In [ ]:
if 'pan' in maps:
    del maps['pan']
maps['azimuth'] = x/xscale
sources = Events(maps.keys())
sources.fromdict(maps)
soni = Sonification(score, sources, copy.copy(synth), system)
sources.apply_mapping_functions()
soni.render()
dobj = soni.notebook_display(show_waveform=0)

There is a new `angle_unit` optional keyword to `apply_mapping_functions`. To guide the user, if no `map_lims` value or `angle_unit` is set it will warn as above. If we changed to radians, the same x values only traverse a sixth of a circle: 

In [ ]:
if 'pan' in maps:
    del maps['pan']
maps['azimuth'] = x/xscale
sources = Events(maps.keys())
sources.fromdict(maps)
soni = Sonification(score, sources, copy.copy(synth), system)
sources.apply_mapping_functions(angle_unit='radians')
soni.render()
dobj = soni.notebook_display(show_waveform=0)

`degrees`, `radians` and `cycles` are supported as units. The user can also use `map_lims` to set a custom input range as before.